# LLM Tool Execution Workflow

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

### Tool Creation

In [3]:
# Creating a tool
@tool
def multiply(a:int, b:int) -> int:
    """Returns the multiplication product of two numbers"""
    return a*b

In [4]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

### Tool Binding
Linking tool with LLM

In [5]:
# Tool Binding
# Selecting a Model
llm = ChatOpenAI()

# Bind all the tools (function name) using bind_tools() function in LLM Runnable
# Pass the list of tools in the function you want to binf with LLM
llm_with_tool = llm.bind_tools([multiply])

### Tool Calling
Picking correct tool for execution and defining the input variable for tool

In [6]:
# Normal LLM Response
result1 = llm_with_tool.invoke("How are you?")

In [7]:
print("LLM Output",result1)
print("\n\nOutput generated by LLM: ",result1.content)
print("\n\nTool call by LLM: ",result1.tool_calls)

LLM Output content="I'm here and ready to help you with any questions or tasks you have. What can I assist you with today?" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 53, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWpYZMkRCkHOTsjmqgUas0uTC0RH0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dac96-ac53-7e11-8bfc-c6ef0d25938e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 25, 'total_tokens': 78, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


Output generated by LLM:  I'm here and ready to 

In [8]:
# Tool calling
result2 = llm_with_tool.invoke("HMultiply 2 and 6")

In [9]:
print("LLM Output",result2)
print("\n\nOutput generated by LLM: ",result2.content)
print("\n\nTool call by LLM: ",result2.tool_calls)

LLM Output content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 56, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWpYaNt9OQTfh4LBiFgHwLx3lelbI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019dac96-b208-7372-b302-96ea4e8c78c9-0' tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 6}, 'id': 'call_gbl83mQg7hsblMjDGESmKwQL', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 56, 'output_tokens': 32, 'total_tokens': 88, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


Output generated by LLM:  


Tool call

### Tool Execution
Execute the tool to get output result

In [10]:
# Actual output from LLM in tool_calls argument
result2.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 6},
  'id': 'call_gbl83mQg7hsblMjDGESmKwQL',
  'type': 'tool_call'}]

In [11]:
# Trimming to required info to pass into the tool
result2.tool_calls[0]['args']

{'a': 2, 'b': 6}

In [12]:
# Executing the tool
multiply.invoke(result2.tool_calls[0]['args'])

12

In [13]:
# Passing the whole tools_calls output in multiply tool
# It will generate a ToolMessage
# ToolMessage is different from AIMessage, HumanMessage and SystemMessage
# It helps the LLM to identify the LLM the Tool has also provided a output.
# LLM will read the tool message and provide the output the user
multiply.invoke(result2.tool_calls[0])

ToolMessage(content='12', name='multiply', tool_call_id='call_gbl83mQg7hsblMjDGESmKwQL')

### Creating a proper workflow

In [14]:
# Creating a chat history
history = []

In [15]:
# Wrapping a query inside HumanMessage and adding in history
message = 'Show me the multiplication product od 10 and 3'
query  = HumanMessage(message)
history.append(query)
history

[HumanMessage(content='Show me the multiplication product od 10 and 3', additional_kwargs={}, response_metadata={})]

In [16]:
# Asking the query to LLM
message = llm_with_tool.invoke(history)
message.tool_calls[0]


{'name': 'multiply',
 'args': {'a': 10, 'b': 3},
 'id': 'call_OihqodNKu8em1wJY4GI2TSYm',
 'type': 'tool_call'}

In [17]:
# Saving the output from LLM in history
history.append(message)
history

[HumanMessage(content='Show me the multiplication product od 10 and 3', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 60, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWpYddvc4U3ThuLbIMhUsAdOswVDn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dac96-c0d7-7cc1-b956-a305b7989ddd-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 3}, 'id': 'call_OihqodNKu8em1wJY4GI2TSYm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 32, 'total_tokens': 92, 'input_token_details': {

In [18]:
# Invoking the tool
message = multiply.invoke(message.tool_calls[0])
message

ToolMessage(content='30', name='multiply', tool_call_id='call_OihqodNKu8em1wJY4GI2TSYm')

In [19]:
# Appending the Tool message to history
history.append(message)
history

[HumanMessage(content='Show me the multiplication product od 10 and 3', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 60, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWpYddvc4U3ThuLbIMhUsAdOswVDn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dac96-c0d7-7cc1-b956-a305b7989ddd-0', tool_calls=[{'name': 'multiply', 'args': {'a': 10, 'b': 3}, 'id': 'call_OihqodNKu8em1wJY4GI2TSYm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 32, 'total_tokens': 92, 'input_token_details': {

In [20]:
message = llm_with_tool.invoke(history)

In [21]:
message.content

'The multiplication product of 10 and 3 is 30.'

### Single Code Block to make it work

In [26]:
result = ''
history = []
while True:
    user_input = input("You: ")
    history.append(HumanMessage(user_input))
    if user_input == 'exit':
        break
    result = llm_with_tool.invoke(history)
    history.append(result)
    if result.content == '':
        product = multiply.invoke(result.tool_calls[0])
        history.append(product)
        result = llm_with_tool.invoke(history)
        history.append(result)
    print("AI: ",result.content)


AI:  Hello! How can I assist you today?
AI:  I'm here to help you with any questions or tasks you have. Just let me know what you need assistance with!
AI:  The product of multiplying 2 with 500 is 1000.
AI:  Goodbye! Don't hesitate to return if you need further assistance. Have a great day!


In [27]:
history

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 50, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWpabshYcBXDKnhRJxq5UqsTprHIo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dac98-9aa9-7823-baa6-e734519f7f31-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 10, 'total_tokens': 60, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='What are you d